In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.grid"] = True

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

In [ ]:
X, y = make_regression(
    n_samples=320,
    n_features=2,
    n_informative=2,
    noise=8.0,
    bias=25.0,
    random_state=42
)

X = X.astype(np.float32)
y = y.astype(np.float32)

X[:, 0] = X[:, 0] * 40.0
X[:, 1] = X[:, 1] * 1.0

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

dataset_info = pd.DataFrame({
    "Feature": ["x1", "x2"],
    "Mean": X_train.mean(axis=0),
    "Std": X_train.std(axis=0),
    "Min": X_train.min(axis=0),
    "Max": X_train.max(axis=0)
})

display(dataset_info)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc1 = axes[0].scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=y_train,
    cmap="turbo",
    s=55,
    alpha=0.88,
    edgecolors="white",
    linewidths=0.5
)

axes[0].set_title("Synthetic Dataset")
axes[0].set_xlabel("Feature x1")
axes[0].set_ylabel("Feature x2")
fig.colorbar(sc1, ax=axes[0], label="Target y")

axes[1].hist(y_train, bins=28, color="#7b2cbf", alpha=0.85, edgecolor="white")
axes[1].set_title("Target Distribution")
axes[1].set_xlabel("Target y")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

scaled_info = pd.DataFrame({
    "Feature": ["x1", "x2"],
    "Mean before": X_train.mean(axis=0),
    "Std before": X_train.std(axis=0),
    "Mean after": X_train_scaled.mean(axis=0),
    "Std after": X_train_scaled.std(axis=0)
})

display(scaled_info)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=y_train,
    cmap="turbo",
    s=50,
    alpha=0.85
)
axes[0].set_title("Before Standardization")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")

axes[1].scatter(
    X_train_scaled[:, 0],
    X_train_scaled[:, 1],
    c=y_train,
    cmap="turbo",
    s=50,
    alpha=0.85
)
axes[1].set_title("After Standardization")
axes[1].set_xlabel("x1 scaled")
axes[1].set_ylabel("x2 scaled")

plt.tight_layout()
plt.show()

In [ ]:
def loss_surface(w1, w2):
    return 0.5 * (w1**2 + 25.0 * w2**2) + 0.15 * np.sin(1.2 * w1) * np.sin(w2)

def loss_value(point):
    w1, w2 = point
    return loss_surface(w1, w2)

def gradient_value(point):
    w1, w2 = point
    dw1 = w1 + 0.18 * np.cos(1.2 * w1) * np.sin(w2)
    dw2 = 25.0 * w2 + 0.15 * np.sin(1.2 * w1) * np.cos(w2)
    return np.array([dw1, dw2], dtype=float)

x = np.linspace(-6, 6, 260)
y = np.linspace(-2.4, 2.4, 220)

Xg, Yg = np.meshgrid(x, y)
Zg = loss_surface(Xg, Yg)

minimum_index = np.unravel_index(np.argmin(Zg), Zg.shape)
minimum_point = np.array([Xg[minimum_index], Yg[minimum_index]])

print("Approximate visual minimum:", minimum_point)
print("Minimum loss:", Zg[minimum_index])

In [ ]:
fig = plt.figure(figsize=(15, 9))
ax = fig.add_subplot(111, projection="3d")

surface = ax.plot_surface(
    Xg,
    Yg,
    Zg,
    cmap="turbo",
    alpha=0.76,
    linewidth=0
)

ax.scatter(
    minimum_point[0],
    minimum_point[1],
    loss_value(minimum_point),
    s=170,
    marker="*",
    color="red",
    edgecolors="black"
)

ax.set_title("3D Ill-Conditioned Loss Surface")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_zlabel("Loss")
fig.colorbar(surface, ax=ax, shrink=0.60, pad=0.10, label="Loss")
ax.view_init(elev=37, azim=-55)

plt.show()

fig, ax = plt.subplots(figsize=(13, 9))

levels = np.linspace(
    np.percentile(Zg, 2),
    np.percentile(Zg, 96),
    45
)

cf = ax.contourf(
    Xg,
    Yg,
    Zg,
    levels=levels,
    cmap="turbo"
)

ax.contour(
    Xg,
    Yg,
    Zg,
    levels=levels[::3],
    colors="white",
    linewidths=0.55,
    alpha=0.35
)

ax.scatter(
    minimum_point[0],
    minimum_point[1],
    s=150,
    marker="*",
    color="red"
)

ax.set_title("Contour Map of the Same Loss Surface")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_aspect("equal")
plt.colorbar(cf, ax=ax, label="Loss")

plt.show()

In [ ]:
def optimize_sgd(start, lr=0.018, steps=80):
    w = np.array(start, dtype=float)
    path = [w.copy()]
    losses = [loss_value(w)]
    gradients = []
    effective_lrs = [np.array([lr, lr], dtype=float)]

    for _ in range(steps):
        g = gradient_value(w)
        w = w - lr * g

        path.append(w.copy())
        losses.append(loss_value(w))
        gradients.append(g.copy())
        effective_lrs.append(np.array([lr, lr], dtype=float))

    return np.array(path), np.array(losses), np.array(gradients), np.array(effective_lrs)

def optimize_adagrad(start, lr=0.70, epsilon=1e-8, steps=80):
    w = np.array(start, dtype=float)
    accumulator = np.zeros(2, dtype=float)

    path = [w.copy()]
    losses = [loss_value(w)]
    gradients = []
    effective_lrs = [np.array([lr, lr], dtype=float)]
    accumulators = [accumulator.copy()]

    for _ in range(steps):
        g = gradient_value(w)
        accumulator = accumulator + g**2
        effective_lr = lr / (np.sqrt(accumulator) + epsilon)
        w = w - effective_lr * g

        path.append(w.copy())
        losses.append(loss_value(w))
        gradients.append(g.copy())
        effective_lrs.append(effective_lr.copy())
        accumulators.append(accumulator.copy())

    return (
        np.array(path),
        np.array(losses),
        np.array(gradients),
        np.array(effective_lrs),
        np.array(accumulators)
    )

start = np.array([-5.0, 2.0])

sgd_path, sgd_losses, sgd_gradients, sgd_lrs = optimize_sgd(start)
adagrad_path, adagrad_losses, adagrad_gradients, adagrad_lrs, adagrad_acc = optimize_adagrad(start)

print("SGD final loss:", sgd_losses[-1])
print("Adagrad final loss:", adagrad_losses[-1])
print("SGD final point:", sgd_path[-1])
print("Adagrad final point:", adagrad_path[-1])

In [ ]:
summary_surface = pd.DataFrame({
    "Optimizer": ["SGD", "Adagrad"],
    "Initial Loss": [sgd_losses[0], adagrad_losses[0]],
    "Final Loss": [sgd_losses[-1], adagrad_losses[-1]],
    "Best Loss": [sgd_losses.min(), adagrad_losses.min()],
    "Final w1": [sgd_path[-1, 0], adagrad_path[-1, 0]],
    "Final w2": [sgd_path[-1, 1], adagrad_path[-1, 1]]
})

summary_surface

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

levels = np.linspace(
    np.percentile(Zg, 2),
    np.percentile(Zg, 96),
    45
)

cf = ax.contourf(
    Xg,
    Yg,
    Zg,
    levels=levels,
    cmap="turbo"
)

ax.contour(
    Xg,
    Yg,
    Zg,
    levels=levels[::3],
    colors="white",
    linewidths=0.5,
    alpha=0.28
)

ax.plot(
    sgd_path[:, 0],
    sgd_path[:, 1],
    linewidth=3,
    label="SGD",
    color="#ff4d6d"
)

ax.plot(
    adagrad_path[:, 0],
    adagrad_path[:, 1],
    linewidth=3,
    label="Adagrad",
    color="#00b4d8"
)

ax.scatter(
    *start,
    s=150,
    marker="*",
    color="black",
    label="Start",
    zorder=6
)

ax.scatter(
    *minimum_point,
    s=150,
    marker="X",
    color="#39ff14",
    label="Minimum",
    zorder=6
)

ax.scatter(
    sgd_path[-1, 0],
    sgd_path[-1, 1],
    s=110,
    marker="o",
    color="#ff4d6d",
    label="SGD End"
)

ax.scatter(
    adagrad_path[-1, 0],
    adagrad_path[-1, 1],
    s=110,
    marker="o",
    color="#00b4d8",
    label="Adagrad End"
)

ax.set_title("SGD vs Adagrad Travel Paths on the Contour Surface")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_aspect("equal")
ax.legend()
plt.colorbar(cf, ax=ax, label="Loss")

plt.show()

In [ ]:
stride = 4
Xs = Xg[::stride, ::stride]
Ys = Yg[::stride, ::stride]
Zs = Zg[::stride, ::stride]

fig = plt.figure(figsize=(17, 10))
ax = fig.add_subplot(111, projection="3d")

surface = ax.plot_surface(
    Xs,
    Ys,
    Zs,
    cmap="turbo",
    alpha=0.68,
    linewidth=0
)

ax.plot(
    sgd_path[:, 0],
    sgd_path[:, 1],
    [loss_value(p) for p in sgd_path],
    linewidth=3.3,
    label="SGD",
    color="#ff4d6d"
)

ax.plot(
    adagrad_path[:, 0],
    adagrad_path[:, 1],
    [loss_value(p) for p in adagrad_path],
    linewidth=3.3,
    label="Adagrad",
    color="#00b4d8"
)

ax.scatter(
    start[0],
    start[1],
    loss_value(start),
    s=130,
    marker="*",
    color="black"
)

ax.scatter(
    minimum_point[0],
    minimum_point[1],
    loss_value(minimum_point),
    s=150,
    marker="X",
    color="#39ff14"
)

ax.set_title("3D Optimization Travel: SGD vs Adagrad")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_zlabel("Loss")
fig.colorbar(surface, ax=ax, shrink=0.55, pad=0.08, label="Loss")
ax.legend()
ax.view_init(elev=38, azim=-60)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))

levels = np.linspace(
    np.percentile(Zg, 2),
    np.percentile(Zg, 96),
    45
)

cf = ax.contourf(
    Xg,
    Yg,
    Zg,
    levels=levels,
    cmap="turbo"
)

ax.contour(
    Xg,
    Yg,
    Zg,
    levels=levels[::3],
    colors="white",
    linewidths=0.5,
    alpha=0.28
)

line_sgd, = ax.plot([], [], linewidth=3, color="#ff4d6d", label="SGD")
line_adagrad, = ax.plot([], [], linewidth=3, color="#00b4d8", label="Adagrad")

point_sgd, = ax.plot(
    [], [],
    marker="o",
    markersize=9,
    color="#ff4d6d"
)

point_adagrad, = ax.plot(
    [], [],
    marker="o",
    markersize=9,
    color="#00b4d8"
)

ax.scatter(
    *start,
    s=140,
    marker="*",
    color="black",
    zorder=5,
    label="Start"
)

ax.scatter(
    *minimum_point,
    s=140,
    marker="X",
    color="#39ff14",
    zorder=5,
    label="Minimum"
)

ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_title("Animated Optimizer Travel — SGD vs Adagrad")
ax.set_aspect("equal")
ax.legend()
plt.colorbar(cf, ax=ax, label="Loss")

step_text = ax.text(
    0.02,
    0.96,
    "",
    transform=ax.transAxes,
    fontsize=13,
    bbox=dict(boxstyle="round", facecolor="black", alpha=0.72),
    color="white"
)

def init_animation():
    for artist in [line_sgd, line_adagrad, point_sgd, point_adagrad]:
        artist.set_data([], [])
    step_text.set_text("")
    return line_sgd, line_adagrad, point_sgd, point_adagrad, step_text

def update_animation(frame):
    n = frame + 1

    line_sgd.set_data(
        sgd_path[:n, 0],
        sgd_path[:n, 1]
    )

    line_adagrad.set_data(
        adagrad_path[:n, 0],
        adagrad_path[:n, 1]
    )

    point_sgd.set_data(
        [sgd_path[n - 1, 0]],
        [sgd_path[n - 1, 1]]
    )

    point_adagrad.set_data(
        [adagrad_path[n - 1, 0]],
        [adagrad_path[n - 1, 1]]
    )

    step_text.set_text(
        f"Step {frame}\n"
        f"SGD loss: {sgd_losses[n - 1]:.3f}\n"
        f"Adagrad loss: {adagrad_losses[n - 1]:.3f}"
    )

    return line_sgd, line_adagrad, point_sgd, point_adagrad, step_text

anim = FuncAnimation(
    fig,
    update_animation,
    init_func=init_animation,
    frames=len(sgd_path),
    interval=90,
    blit=True,
    repeat=False
)

plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
fig = plt.figure(figsize=(17, 10))
ax = fig.add_subplot(111, projection="3d")

surface = ax.plot_surface(
    Xs,
    Ys,
    Zs,
    cmap="turbo",
    alpha=0.62,
    linewidth=0
)

line_sgd3d, = ax.plot([], [], [], linewidth=3.3, label="SGD", color="#ff4d6d")
line_adagrad3d, = ax.plot([], [], [], linewidth=3.3, label="Adagrad", color="#00b4d8")

point_sgd3d, = ax.plot([], [], [], marker="o", markersize=8, color="#ff4d6d")
point_adagrad3d, = ax.plot([], [], [], marker="o", markersize=8, color="#00b4d8")

ax.scatter(
    start[0],
    start[1],
    loss_value(start),
    s=140,
    marker="*",
    color="black"
)

ax.scatter(
    minimum_point[0],
    minimum_point[1],
    loss_value(minimum_point),
    s=150,
    marker="X",
    color="#39ff14"
)

ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_zlabel("Loss")
ax.set_title("3D Animated Descent — SGD vs Adagrad")
fig.colorbar(surface, ax=ax, shrink=0.55, pad=0.08, label="Loss")
ax.legend()
ax.view_init(elev=36, azim=-60)

sgd_z = np.array([loss_value(p) for p in sgd_path])
adagrad_z = np.array([loss_value(p) for p in adagrad_path])

def init_3d():
    for artist in [
        line_sgd3d,
        line_adagrad3d,
        point_sgd3d,
        point_adagrad3d
    ]:
        artist.set_data([], [])
        artist.set_3d_properties([])
    return line_sgd3d, line_adagrad3d, point_sgd3d, point_adagrad3d

def update_3d(frame):
    n = frame + 1

    line_sgd3d.set_data(
        sgd_path[:n, 0],
        sgd_path[:n, 1]
    )
    line_sgd3d.set_3d_properties(sgd_z[:n])

    point_sgd3d.set_data(
        [sgd_path[n - 1, 0]],
        [sgd_path[n - 1, 1]]
    )
    point_sgd3d.set_3d_properties([sgd_z[n - 1]])

    line_adagrad3d.set_data(
        adagrad_path[:n, 0],
        adagrad_path[:n, 1]
    )
    line_adagrad3d.set_3d_properties(adagrad_z[:n])

    point_adagrad3d.set_data(
        [adagrad_path[n - 1, 0]],
        [adagrad_path[n - 1, 1]]
    )
    point_adagrad3d.set_3d_properties([adagrad_z[n - 1]])

    ax.set_title(f"3D Animated Descent — Step {frame}")

    return line_sgd3d, line_adagrad3d, point_sgd3d, point_adagrad3d

anim3d = FuncAnimation(
    fig,
    update_3d,
    init_func=init_3d,
    frames=len(sgd_path),
    interval=90,
    blit=True,
    repeat=False
)

plt.close(fig)
HTML(anim3d.to_jshtml())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

axes[0].plot(
    sgd_lrs[:, 0],
    linewidth=2.8,
    label="w1"
)
axes[0].plot(
    sgd_lrs[:, 1],
    linewidth=2.8,
    linestyle="--",
    label="w2"
)
axes[0].set_title("SGD Effective Learning Rates")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Effective Learning Rate")
axes[0].legend()

axes[1].plot(
    adagrad_lrs[:, 0],
    linewidth=2.8,
    label="w1"
)
axes[1].plot(
    adagrad_lrs[:, 1],
    linewidth=2.8,
    linestyle="--",
    label="w2"
)
axes[1].set_title("Adagrad Parameter-wise Effective Learning Rates")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Effective Learning Rate")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

axes[0].plot(
    sgd_losses,
    linewidth=2.8,
    color="#ff4d6d",
    label="SGD"
)
axes[0].plot(
    adagrad_losses,
    linewidth=2.8,
    color="#00b4d8",
    label="Adagrad"
)
axes[0].set_title("Training Loss on Visual Objective")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].semilogy(
    sgd_losses + 1e-8,
    linewidth=2.8,
    color="#ff4d6d",
    label="SGD"
)
axes[1].semilogy(
    adagrad_losses + 1e-8,
    linewidth=2.8,
    color="#00b4d8",
    label="Adagrad"
)
axes[1].set_title("Training Loss — Log Scale")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
def build_ann():
    return keras.Sequential([
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1)
    ])

tf.keras.backend.clear_session()
tf.random.set_seed(123)

base_model = build_ann()
initial_weights = base_model.get_weights()

sgd_model = build_ann()
adagrad_model = build_ann()

sgd_model.set_weights(initial_weights)
adagrad_model.set_weights(initial_weights)

sgd_optimizer = keras.optimizers.SGD(
    learning_rate=0.01,
    momentum=0.0,
    clipnorm=1.0
)

adagrad_optimizer = keras.optimizers.Adagrad(
    learning_rate=0.005,
    epsilon=1e-7,
    clipnorm=1.0
)

sgd_model.compile(
    optimizer=sgd_optimizer,
    loss="mse",
    metrics=["mae"]
)

adagrad_model.compile(
    optimizer=adagrad_optimizer,
    loss="mse",
    metrics=["mae"]
)

print("SGD configuration:")
print(sgd_optimizer.get_config())

print("\nAdagrad configuration:")
print(adagrad_optimizer.get_config())

In [ ]:
class WeightPathRecorder(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.weights = []
        self.losses = []
        self.val_losses = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        flat = np.concatenate([
            w.numpy().ravel()
            for w in self.model.trainable_weights
        ])
        self.weights.append(flat.copy())
        self.losses.append(float(logs.get("loss", np.nan)))
        self.val_losses.append(float(logs.get("val_loss", np.nan)))

sgd_recorder = WeightPathRecorder()
adagrad_recorder = WeightPathRecorder()

sgd_history = sgd_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    shuffle=False,
    verbose=0,
    callbacks=[sgd_recorder]
)

adagrad_history = adagrad_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    shuffle=False,
    verbose=0,
    callbacks=[adagrad_recorder]
)

print("Training complete")

In [ ]:
ann_loss = pd.DataFrame({
    "Epoch": np.arange(1, 101),
    "SGD": sgd_history.history["loss"],
    "Adagrad": adagrad_history.history["loss"]
})

ann_val_loss = pd.DataFrame({
    "Epoch": np.arange(1, 101),
    "SGD": sgd_history.history["val_loss"],
    "Adagrad": adagrad_history.history["val_loss"]
})

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

axes[0].plot(
    ann_loss["Epoch"],
    ann_loss["SGD"],
    linewidth=2.7,
    color="#ff4d6d",
    label="SGD"
)
axes[0].plot(
    ann_loss["Epoch"],
    ann_loss["Adagrad"],
    linewidth=2.7,
    color="#00b4d8",
    label="Adagrad"
)
axes[0].set_title("ANN Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()

axes[1].plot(
    ann_val_loss["Epoch"],
    ann_val_loss["SGD"],
    linewidth=2.7,
    color="#ff4d6d",
    label="SGD"
)
axes[1].plot(
    ann_val_loss["Epoch"],
    ann_val_loss["Adagrad"],
    linewidth=2.7,
    color="#00b4d8",
    label="Adagrad"
)
axes[1].set_title("ANN Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
pred_sgd = sgd_model.predict(X_test_scaled, verbose=0).ravel()
pred_adagrad = adagrad_model.predict(X_test_scaled, verbose=0).ravel()

metrics = pd.DataFrame({
    "Optimizer": ["SGD", "Adagrad"],
    "MAE": [
        mean_absolute_error(y_test, pred_sgd),
        mean_absolute_error(y_test, pred_adagrad)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, pred_sgd)),
        np.sqrt(mean_squared_error(y_test, pred_adagrad))
    ],
    "R2": [
        r2_score(y_test, pred_sgd),
        r2_score(y_test, pred_adagrad)
    ],
    "Final Train MSE": [
        sgd_history.history["loss"][-1],
        adagrad_history.history["loss"][-1]
    ],
    "Final Val MSE": [
        sgd_history.history["val_loss"][-1],
        adagrad_history.history["val_loss"][-1]
    ]
})

metrics.round(4)

In [ ]:
from sklearn.decomposition import PCA

all_weights = np.vstack([
    np.vstack(sgd_recorder.weights),
    np.vstack(adagrad_recorder.weights)
])

pca = PCA(n_components=2)
all_pc = pca.fit_transform(all_weights)

n_epochs = len(sgd_recorder.weights)

sgd_pc = all_pc[:n_epochs]
adagrad_pc = all_pc[n_epochs:]

sgd_loss_arr = np.array(sgd_recorder.losses)
adagrad_loss_arr = np.array(adagrad_recorder.losses)

fig = plt.figure(figsize=(17, 10))
ax = fig.add_subplot(111, projection="3d")

ax.plot(
    sgd_pc[:, 0],
    sgd_pc[:, 1],
    sgd_loss_arr,
    linewidth=3,
    label="SGD",
    color="#ff4d6d"
)

ax.plot(
    adagrad_pc[:, 0],
    adagrad_pc[:, 1],
    adagrad_loss_arr,
    linewidth=3,
    label="Adagrad",
    color="#00b4d8"
)

ax.scatter(
    sgd_pc[0, 0],
    sgd_pc[0, 1],
    sgd_loss_arr[0],
    s=120,
    marker="*",
    color="black"
)

ax.set_title("3D ANN Weight-Space Training Trajectories")
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_zlabel("Training MSE")
ax.legend()
ax.view_init(elev=30, azim=-60)

plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
fig = plt.figure(figsize=(17, 10))
ax = fig.add_subplot(111, projection="3d")

ax.plot(
    sgd_pc[:, 0],
    sgd_pc[:, 1],
    sgd_loss_arr,
    linewidth=3,
    label="SGD",
    color="#ff4d6d"
)

ax.plot(
    adagrad_pc[:, 0],
    adagrad_pc[:, 1],
    adagrad_loss_arr,
    linewidth=3,
    label="Adagrad",
    color="#00b4d8"
)

point_sgd_ann, = ax.plot(
    [],
    [],
    [],
    marker="o",
    markersize=8,
    color="#ff4d6d"
)

point_adagrad_ann, = ax.plot(
    [],
    [],
    [],
    marker="o",
    markersize=8,
    color="#00b4d8"
)

ax.set_xlim(
    all_pc[:, 0].min(),
    all_pc[:, 0].max()
)

ax.set_ylim(
    all_pc[:, 1].min(),
    all_pc[:, 1].max()
)

ax.set_zlim(
    min(sgd_loss_arr.min(), adagrad_loss_arr.min()),
    max(sgd_loss_arr.max(), adagrad_loss_arr.max())
)

ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_zlabel("Training MSE")
ax.set_title("Animated 3D ANN Optimization Trajectory")
ax.view_init(elev=30, azim=-60)
ax.legend()

def init_ann_3d():
    for artist in [point_sgd_ann, point_adagrad_ann]:
        artist.set_data([], [])
        artist.set_3d_properties([])
    return point_sgd_ann, point_adagrad_ann

def update_ann_3d(frame):
    point_sgd_ann.set_data(
        [sgd_pc[frame, 0]],
        [sgd_pc[frame, 1]]
    )
    point_sgd_ann.set_3d_properties(
        [sgd_loss_arr[frame]]
    )

    point_adagrad_ann.set_data(
        [adagrad_pc[frame, 0]],
        [adagrad_pc[frame, 1]]
    )
    point_adagrad_ann.set_3d_properties(
        [adagrad_loss_arr[frame]]
    )

    ax.set_title(
        f"Animated ANN Optimization — Epoch {frame + 1}"
    )

    return point_sgd_ann, point_adagrad_ann

ann_anim = FuncAnimation(
    fig,
    update_ann_3d,
    init_func=init_ann_3d,
    frames=n_epochs,
    interval=100,
    blit=True,
    repeat=False
)

plt.close(fig)
HTML(ann_anim.to_jshtml())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

axes[0].plot(
    np.arange(1, n_epochs + 1),
    sgd_loss_arr,
    linewidth=2.7,
    color="#ff4d6d",
    label="SGD"
)

axes[0].plot(
    np.arange(1, n_epochs + 1),
    adagrad_loss_arr,
    linewidth=2.7,
    color="#00b4d8",
    label="Adagrad"
)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training MSE")
axes[0].set_title("Optimizer Convergence")
axes[0].legend()

axes[1].semilogy(
    np.arange(1, n_epochs + 1),
    sgd_loss_arr + 1e-8,
    linewidth=2.7,
    color="#ff4d6d",
    label="SGD"
)

axes[1].semilogy(
    np.arange(1, n_epochs + 1),
    adagrad_loss_arr + 1e-8,
    linewidth=2.7,
    color="#00b4d8",
    label="Adagrad"
)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Training MSE")
axes[1].set_title("Optimizer Convergence — Log Scale")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "Quantity": [
        "Uses current gradient",
        "Stores squared-gradient history",
        "Single global learning rate",
        "Parameter-wise effective learning rate",
        "Learning-rate behavior",
        "Typical strength",
        "Typical weakness"
    ],
    "SGD": [
        "Yes",
        "No",
        "Yes",
        "No",
        "Constant",
        "Simple and predictable baseline",
        "Sensitive to learning-rate choice"
    ],
    "Adagrad": [
        "Yes",
        "Yes",
        "No",
        "Yes",
        "Shrinks over time",
        "Useful when gradient scales differ",
        "Can become too conservative"
    ]
})

comparison

In [ ]:
best_rmse = metrics.loc[metrics["RMSE"].idxmin(), "Optimizer"]
best_mae = metrics.loc[metrics["MAE"].idxmin(), "Optimizer"]
best_r2 = metrics.loc[metrics["R2"].idxmax(), "Optimizer"]

print("Best RMSE optimizer:", best_rmse)
print("Best MAE optimizer:", best_mae)
print("Best R² optimizer:", best_r2)
print()
print(metrics.round(4).to_string(index=False))